# 🚀 内存管理（Part 1：Sessions）

**欢迎来到 Kaggle 5 天 Agents 课程 Day 3！**

在本 Notebook 中，你将学习：

- ✅ 什么是 session，以及如何在 Agent 中使用它
- ✅ 如何基于 session 与 events 构建*有状态（stateful）* Agent
- ✅ 如何把 session 持久化到数据库
- ✅ 上下文管理实践（如 context compaction）
- ✅ 共享 session state 的最佳实践

---
## ⚙️ 第 1 部分：环境准备

### 1.1：安装依赖

Kaggle Notebooks 环境已预装 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其依赖，因此本 Notebook 无需额外安装。

如果你要在课程外、自己的 Python 环境中安装并使用 ADK，可运行：

```
pip install google-adk
```

### 1.2：配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/docs)，需要先鉴权。

**1. 获取 API key**

如果你还没有，可在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 key 添加到 Kaggle Secrets**

接下来，把 API key 作为 Kaggle User Secret 添加到 Notebook。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons` -> `Secrets`。
2. 新建标签为 `GOOGLE_API_KEY` 的 secret。
3. 把 API key 粘贴到 "Value" 字段并点击 "Save"。
4. 确认 `GOOGLE_API_KEY` 左侧复选框已勾选，使其附加到该 Notebook。

**3. 在 Notebook 中鉴权**

运行下面代码单元完成鉴权。

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


### 1.3：导入 ADK 组件

现在导入 Agent Development Kit 与 Generative AI 的关键组件。这样代码结构更清晰，也确保我们具备必要构建块。

In [3]:
from typing import Any, Dict

from google.adk.agents import Agent, LlmAgent
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.models.google_llm import Gemini
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4：辅助函数

这个辅助函数用于管理完整会话流程：创建/读取 session、处理用户 query、流式输出响应。支持单条 query，也支持按顺序处理多条 query。

示例：

```
>>> await run_session(runner, "What is the capital of France?", "geography-session")
>>> await run_session(runner, ["Hello!", "What's my name?"], "user-intro-session")
```

In [4]:
# Define helper functions that will be reused throughout the notebook
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
):
    print(f"\n ### Session: {session_name}")

    # Get app name from the Runner
    app_name = runner_instance.app_name

    # Attempt to create a new session or retrieve an existing one
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
    except:
        session = await session_service.get_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )

    # Process queries if provided
    if user_queries:
        # Convert single query to list for uniform processing
        if type(user_queries) == str:
            user_queries = [user_queries]

        # Process each query in the list sequentially
        for query in user_queries:
            print(f"\nUser > {query}")

            # Convert the query string to the ADK Content format
            query = types.Content(role="user", parts=[types.Part(text=query)])

            # Stream the agent's response asynchronously
            async for event in runner_instance.run_async(
                user_id=USER_ID, session_id=session.id, new_message=query
            ):
                # Check if the event contains valid content
                if event.content and event.content.parts:
                    # Filter out empty or "None" responses before printing
                    if (
                        event.content.parts[0].text != "None"
                        and event.content.parts[0].text
                    ):
                        print(f"{MODEL_NAME} > ", event.content.parts[0].text)
    else:
        print("No queries!")


print("✅ Helper functions defined.")

✅ Helper functions defined.


### 1.5：配置重试选项

在使用 LLM 时，你可能会遇到速率限制或服务暂时不可用等瞬时错误。重试选项会通过指数退避自动重试请求。

In [5]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

---
## 🤹 第 2 部分：Session 管理

### 2.1 问题背景

Large Language Model 在本质上是**无状态（stateless）**的。它只“知道”单次 API 调用里你给它的内容。也就是说，如果没有上下文管理，Agent 只能根据当前 prompt 响应，而不会考虑历史对话。

**❓ 这为什么重要？** 想象你在和一个“每说完一句就失忆”的人聊天，这就是原始 LLM 的典型挑战。

在 ADK 中，我们用 `Sessions` 管理**短期记忆**，用 `Memory` 管理**长期记忆**。下一份 Notebook 你会重点学习 `Memory`。

### 2.2 什么是 Session？

#### **📦 Session**

Session 是对话容器。它按时间顺序封装会话历史，并记录一次**连续对话**中的所有工具交互与响应。Session 与用户和 Agent 绑定，不会被其他用户共享；同样，一个 Agent 的 session 历史不会与其他 Agent 混用。

在 ADK 中，**Session** 由两个核心部分组成：`Events` 与 `State`。

**📝 Session.Events：**

> Session 是容器，`Events` 则是对话的基本构件。
>
> Events 示例：
> - *User Input*：用户消息（文本、音频、图像等）
> - *Agent Response*：Agent 对用户的回复
> - *Tool Call*：Agent 决定调用外部工具/API
> - *Tool Output*：工具返回的数据，供 Agent 继续推理
    

**{} Session.State：**

> `session.state` 是 Agent 的工作草稿区，用于在对话中存储与更新动态信息。你可以把它看作全局 `{key, value}` 存储，所有 subagents 和 tools 都可访问。

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day3/session-state-and-events.png" width="320" alt="Session state and events">

<!-- ```mermaid
graph TD
    subgraph A["Agentic Application"];
        subgraph U["User"]
            subgraph S1["Session"]
                D1["Session.Events"]
                D2["Session.State"]
            end
        end
    end
``` -->

### 2.3 如何管理 sessions？

一个 Agent 应用可服务多个用户，而每个用户又可能拥有多个 session。

为管理这些 session 与 events，ADK 提供了 **Session Manager** 和 **Runner**：

1. **`SessionService`**：存储层
   - 负责 session 数据的创建、存储、读取
   - 提供多种实现以适配不同需求（内存、数据库、云）

2. **`Runner`**：编排层
   - 管理用户与 Agent 之间的信息流
   - 自动维护对话历史
   - 在幕后处理 Context Engineering

可以这样理解：

- **Session** = 一本笔记本 📓
- **Events** = 单页里的条目 📝
- **SessionService** = 存放笔记本的档案柜 🗄️
- **Runner** = 负责对话流程的助手 🤖

### 2.4 实现我们的第一个有状态 Agent

下面来构建第一个可以“记住上下文、进行连续对话”的 stateful agent。

ADK 提供多种 session 类型以适配不同场景。这里先从最简单的 `InMemorySessionService` 开始：

In [6]:
APP_NAME = "default"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session

MODEL_NAME = "gemini-2.5-flash-lite"


# Step 1: Create the LLM Agent
root_agent = Agent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="A text chatbot",  # Description of the agent's purpose
)

# Step 2: Set up Session Management
# InMemorySessionService stores conversations in RAM (temporary)
session_service = InMemorySessionService()

# Step 3: Create the Runner
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

print("✅ Stateful agent initialized!")
print(f"   - Application: {APP_NAME}")
print(f"   - User: {USER_ID}")
print(f"   - Using: {session_service.__class__.__name__}")

✅ Stateful agent initialized!
   - Application: default
   - User: default
   - Using: InMemorySessionService


### 2.5 测试我们的有状态 Agent

现在来看看 session 的效果！

In [7]:
# Run a conversation with two queries in the same session
# Notice: Both queries are part of the SAME session, so context is maintained
await run_session(
    runner,
    [
        "Hi, I am Sam! What is the capital of United States?",
        "Hello! What is my name?",  # This time, the agent should remember!
    ],
    "stateful-agentic-session",
)


 ### Session: stateful-agentic-session

User > Hi, I am Sam! What is the capital of United States?
gemini-2.5-flash-lite >  Hi Sam! The capital of the United States is Washington, D.C.

User > Hello! What is my name?
gemini-2.5-flash-lite >  Your name is Sam!


🎉 **成功！** Agent 记住了你的名字，因为两次提问都在同一个 session 中。Runner 自动维护了对话历史。

但有个限制：`InMemorySessionService` 是临时存储。**应用一旦停止，历史就会丢失。**

### 🛑（可选）2.6 测试 Agent 的“遗忘”

> 若要验证 Agent 会遗忘上下文，请**重启 kernel**。然后**运行本 Notebook 前面所有单元，但跳过 2.5 中的 `run_session` 单元**。
> 
> 再运行下方单元，你会看到 Agent 不再记得之前的对话内容。

In [8]:
# Run this cell after restarting the kernel. All this history will be gone...
await run_session(
    runner,
    ["What did I ask you about earlier?", "And remind me, what's my name?"],
    "stateful-agentic-session",
)  # Note, we are using same session name


 ### Session: stateful-agentic-session

User > What did I ask you about earlier?
gemini-2.5-flash-lite >  You asked me about the capital of the United States.

User > And remind me, what's my name?
gemini-2.5-flash-lite >  Your name is Sam.


### 问题

Session 信息不持久（即有价值对话会丢失）。这在测试环境中有时有利，但**真实场景里，用户应该可以回溯并继续历史对话**。要做到这一点，必须进行持久化。

---
## 📈 第 3 部分：用 `DatabaseSessionService` 持久化 Sessions

`InMemorySessionService` 非常适合原型开发，但真实应用需要会话跨重启、崩溃、部署后继续可用。下面升级到持久化存储。

### 3.1 选择合适的 SessionService

ADK 为不同需求提供不同实现：

| Service | Use Case | Persistence | Best For |
|---------|----------|-------------|----------|
| **InMemorySessionService** | Development & Testing | ❌ 重启丢失 | 快速原型 |
| **DatabaseSessionService** | 自托管应用 | ✅ 重启保留 | 中小型应用 |
| **Agent Engine Sessions** | GCP 生产环境 | ✅ 全托管 | 企业级规模 |


### 3.2 实现持久化 Sessions

下面使用 SQLite 的 `DatabaseSessionService`。这样在演示中无需单独部署数据库服务，也能获得持久化能力。

先创建一个可与用户对话的 `chatbot_agent`。

In [9]:
# Step 1: Create the same agent (notice we use LlmAgent this time)
chatbot_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="A text chatbot with persistent memory",
)

# Step 2: Switch to DatabaseSessionService
# SQLite database will be created automatically
db_url = "sqlite:///my_agent_data.db"  # Local SQLite file
session_service = DatabaseSessionService(db_url=db_url)

# Step 3: Create a new runner with persistent storage
runner = Runner(agent=chatbot_agent, app_name=APP_NAME, session_service=session_service)

print("✅ Upgraded to persistent sessions!")
print(f"   - Database: my_agent_data.db")
print(f"   - Sessions will survive restarts!")

✅ Upgraded to persistent sessions!
   - Database: my_agent_data.db
   - Sessions will survive restarts!


### 3.3 测试 1：验证持久化

在首次测试中，我们用 session ID `test-db-session-01` 发起新对话：先自我介绍名字是 Sam，再问一个问题；第二轮再让 Agent 回答我们的名字。

因为使用了 `DatabaseSessionService`，Agent 应该能记住名字。

对话后，我们还会直接查看 `my_agent_data.db`（SQLite），确认会话 `events`（用户输入与模型响应）是如何存储的。

In [10]:
await run_session(
    runner,
    ["Hi, I am Sam! What is the capital of the United States?", "Hello! What is my name?"],
    "test-db-session-01",
)


 ### Session: test-db-session-01

User > Hi, I am Sam! What is the capital of the United States?
gemini-2.5-flash-lite >  Hi Sam! The capital of the United States is Washington, D.C.

User > Hello! What is my name?
gemini-2.5-flash-lite >  Your name is Sam.


### 🛑（可选）3.4 测试 2：恢复对话

> ‼️ 现在重复测试，但这次先**停止并重启 Kaggle Notebook kernel**。
>
> 1. 运行本 Notebook 前面所有单元，**但跳过** 3.3 的 `run_session` 单元。
>
> 2. 然后运行下方单元，并使用**相同 session ID**（`test-db-session-01`）。

我们会先问一个新问题，再次询问名字。**因为 session 会从数据库加载，Agent 应继续记得**第一次测试中名字是 Sam。这就是持久化 session 的价值。

In [11]:
await run_session(
    runner,
    ["What is the capital of India?", "Hello! What is my name?"],
    "test-db-session-01",
)


 ### Session: test-db-session-01

User > What is the capital of India?
gemini-2.5-flash-lite >  The capital of India is New Delhi.

User > Hello! What is my name?
gemini-2.5-flash-lite >  Your name is Sam.


### 3.5 验证 session 数据隔离

如前所述，session 是 Agent 与 User 之间的私有对话（不同 session 不共享信息）。下面用新的 session 名 `test-db-session-02` 再运行 `run_session` 来验证。

In [12]:
await run_session(
    runner, ["Hello! What is my name?"], "test-db-session-02"
)  # Note, we are using new session name


 ### Session: test-db-session-02

User > Hello! What is my name?
gemini-2.5-flash-lite >  I do not have access to your personal information, so I do not know your name. I am a large language model, trained by Google.


### 3.6 数据库里是如何存储 events 的？

由于我们使用 sqlite 存储信息，下面快速查看一下数据结构。

In [13]:
import sqlite3

def check_data_in_db():
    with sqlite3.connect("my_agent_data.db") as connection:
        cursor = connection.cursor()
        result = cursor.execute(
            "select app_name, session_id, author, content from events"
        )
        print([_[0] for _ in result.description])
        for each in result.fetchall():
            print(each)


check_data_in_db()

['app_name', 'session_id', 'author', 'content']
('default', 'test-db-session-01', 'user', '{"parts": [{"text": "Hi, I am Sam! What is the capital of the United States?"}], "role": "user"}')
('default', 'test-db-session-01', 'text_chat_bot', '{"parts": [{"text": "Hi Sam! The capital of the United States is Washington, D.C."}], "role": "model"}')
('default', 'test-db-session-01', 'user', '{"parts": [{"text": "Hello! What is my name?"}], "role": "user"}')
('default', 'test-db-session-01', 'text_chat_bot', '{"parts": [{"text": "Your name is Sam."}], "role": "model"}')
('default', 'test-db-session-01', 'user', '{"parts": [{"text": "What is the capital of India?"}], "role": "user"}')
('default', 'test-db-session-01', 'text_chat_bot', '{"parts": [{"text": "The capital of India is New Delhi."}], "role": "model"}')
('default', 'test-db-session-01', 'user', '{"parts": [{"text": "Hello! What is my name?"}], "role": "user"}')
('default', 'test-db-session-01', 'text_chat_bot', '{"parts": [{"text": 

---
## ⏳ 第 4 部分：Context Compaction

如你所见，session 数据库会完整保存所有 events，积累速度很快。对于长而复杂的任务，事件列表会非常大，导致性能下降和成本上升。

如果能自动总结历史会怎样？下面使用 ADK 的 **Context Compaction** 功能，演示**如何自动压缩 Session 中保存的上下文。**

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day3/context-compaction.png" width="1400" alt="Context compaction">

### 4.1 为 Agent 创建 App

要启用这个功能，我们继续使用 3.2 中的 `chatbot_agent`。

第一步是创建 `App` 对象：为其命名并传入 chatbot_agent。

同时配置 Context Compaction。`EventsCompactionConfig` 定义两个关键参数：

- **compaction_interval**：每进行 `n` 次对话后触发一次压缩
- **overlap_size**：保留多少条最近对话用于重叠上下文

然后把这个 app 交给 Runner。

In [14]:
# Re-define our app with Events Compaction enabled
research_app_compacting = App(
    name="research_app_compacting",
    root_agent=chatbot_agent,
    # This is the new part!
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=3,  # Trigger compaction every 3 invocations
        overlap_size=1,  # Keep 1 previous turn for context
    ),
)

db_url = "sqlite:///my_agent_data.db"  # Local SQLite file
session_service = DatabaseSessionService(db_url=db_url)

# Create a new runner for our upgraded app
research_runner_compacting = Runner(
    app=research_app_compacting, session_service=session_service
)


print("✅ Research App upgraded with Events Compaction!")

✅ Research App upgraded with Events Compaction!


/tmp/ipykernel_47/3773147741.py:6: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(


### 4.2 运行演示

现在进行一段足够长的对话，以触发 compaction。运行下方单元时，你看到的输出会像普通对话；但由于我们配置了 `App`，第 3 次调用后会在后台静默执行压缩。

下一步我们会验证它确实发生了。

In [15]:
# Turn 1
await run_session(
    research_runner_compacting,
    "What is the latest news about AI in healthcare?",
    "compaction_demo",
)

# Turn 2
await run_session(
    research_runner_compacting,
    "Are there any new developments in drug discovery?",
    "compaction_demo",
)

# Turn 3 - Compaction should trigger after this turn!
await run_session(
    research_runner_compacting,
    "Tell me more about the second development you found.",
    "compaction_demo",
)

# Turn 4
await run_session(
    research_runner_compacting,
    "Who are the main companies involved in that?",
    "compaction_demo",
)


 ### Session: compaction_demo

User > What is the latest news about AI in healthcare?
gemini-2.5-flash-lite >  Here's a summary of some of the latest news and trends in AI in healthcare, drawing from recent developments and ongoing areas of focus:

**1. Generative AI for Drug Discovery and Development:**

*   **Accelerated Discovery:** Companies are increasingly using generative AI models (like large language models - LLMs) to design novel molecules, predict drug-target interactions, and optimize drug candidates. This promises to significantly speed up the early stages of drug discovery.
*   **Personalized Medicine:** Generative AI is also being explored for designing personalized therapies, tailoring treatments based on an individual's genetic makeup and disease profile.
*   **Clinical Trial Optimization:** AI is being used to identify suitable patient cohorts for clinical trials, predict trial outcomes, and even generate synthetic control arms, potentially reducing trial costs and t

### 4.3 在 Session 历史中验证 Compaction

上面的对话看起来正常，但历史实际上已在后台变化。如何证明？

我们可以检查 session 的 `events` 列表。Compaction **不是删除旧事件，而是用一个包含摘要的新 `Event` 替换旧的冗长上下文。**

In [16]:
# Get the final session state
final_session = await session_service.get_session(
    app_name=research_runner_compacting.app_name,
    user_id=USER_ID,
    session_id="compaction_demo",
)

print("--- Searching for Compaction Summary Event ---")
found_summary = False
for event in final_session.events:
    # Compaction events have a 'compaction' attribute
    if event.actions and event.actions.compaction:
        print("\n✅ SUCCESS! Found the Compaction Event:")
        print(f"  Author: {event.author}")
        print(f"\n Compacted information: {event}")
        found_summary = True
        break

if not found_summary:
    print(
        "\n❌ No compaction event found. Try increasing the number of turns in the demo."
    )

--- Searching for Compaction Summary Event ---

✅ SUCCESS! Found the Compaction Event:
  Author: user

 Compacted information: model_version=None content=None grounding_metadata=None partial=None turn_complete=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='90abc972-ebdd-404c-9435-f5f6ed164b3f' author='user' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction={'start_timestamp': 1773582353.168311, 'end_timestamp': 1773582364.557889, 'compacted_content': {'parts': [{'function_call': None, 'code_execution_result': None, 'executable_code': None, 'file_data': None, 'function_response': None, 'inline_data'

### 4.4 你刚刚完成了什么：自动上下文管理

你已经找到“证据”了！在 session 历史中出现这个特殊摘要 `Event`，就是 compaction 生效的直接结果。

**回顾你刚刚看到的过程：**

1. **静默运行**：你执行的是普通对话，表面上看不出差异。
2. **后台压缩**：因为 `App` 配置了 `EventsCompactionConfig`，ADK `Runner` 会自动监控会话长度，达到阈值后在后台触发摘要压缩。
3. **结果可验证**：通过查看 session events，你看到了 LLM 生成的摘要。该摘要会替代旧的冗长历史，成为后续活跃上下文的一部分。

**在后续轮次中，Agent 将基于这份简洁摘要而不是完整历史继续对话。** 这样可以降低成本、提升性能，并帮助 Agent 聚焦关键内容。

### 4.5 ADK 中更多 Context Engineering 选项

#### 👉 Custom Compaction
本示例使用了 ADK 默认摘要器。对于高级场景，你可以自定义 `SlidingWindowCompactor` 并传入配置，以控制摘要 prompt，甚至换用专门 LLM 执行压缩。详见[官方文档](https://google.github.io/adk-docs/context/compaction/)。

#### 👉 Context Caching
ADK 还提供 **Context Caching**：通过缓存请求数据来降低喂给 LLM 的静态指令 token 体积。详见[这里](https://google.github.io/adk-docs/context/caching/)。

### 问题

虽然我们已经能做 Context Compaction，也能通过数据库恢复 session，但又出现了新挑战：有些**关键用户信息或偏好**需要跨 session 共享。

这类场景下，与其共享完整历史，不如只传递少量关键变量，通常体验更好。下面就来看看怎么做。

---
## 🤝 第 5 部分：使用 Session State

### 5.1 为 Session State 管理创建自定义工具

下面演示如何通过自定义工具手动管理 session state。我们将抽取可迁移特征（例如用户名与国家），并创建工具进行存取。

**为什么选这个示例？**

用户名是典型的可复用信息，因为它：

- 只需输入一次，但会多次被引用
- 应该在整个会话中持续有效
- 属于用户特征，能增强个性化体验

本示例中，我们创建两个工具，把用户名与国家写入/读取 Session State。**注意：所有工具都可访问 `ToolContext`。** 实际中你不必为每个字段都单独写一个工具。

In [17]:
# Define scope levels for state keys (following best practices)
USER_NAME_SCOPE_LEVELS = ("temp", "user", "app")


# This demonstrates how tools can write to session state using tool_context.
# The 'user:' prefix indicates this is user-specific data.
def save_userinfo(
    tool_context: ToolContext, user_name: str, country: str
) -> Dict[str, Any]:
    """
    Tool to record and save user name and country in session state.

    Args:
        user_name: The username to store in session state
        country: The name of the user's country
    """
    # Write to session state using the 'user:' prefix for user data
    tool_context.state["user:name"] = user_name
    tool_context.state["user:country"] = country

    return {"status": "success"}


# This demonstrates how tools can read from session state.
def retrieve_userinfo(tool_context: ToolContext) -> Dict[str, Any]:
    """
    Tool to retrieve user name and country from session state.
    """
    # Read from session state
    user_name = tool_context.state.get("user:name", "Username not found")
    country = tool_context.state.get("user:country", "Country not found")

    return {"status": "success", "user_name": user_name, "country": country}


print("✅ Tools created.")

✅ Tools created.


**关键概念：**
- 工具可通过 `tool_context.state` 读写 session state
- 用清晰 key 前缀（`user:`、`app:`、`temp:`）便于组织
- 同一 session 内 state 会跨轮次持续存在

### 5.2 创建带 Session State 工具的 Agent

现在创建一个可访问这些 session state 工具的新 Agent：

In [18]:
# Configuration
APP_NAME = "default"
USER_ID = "default"
MODEL_NAME = "gemini-2.5-flash-lite"

# Create an agent with session state tools
root_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="""A text chatbot.
    Tools for managing user context:
    * To record username and country when provided use `save_userinfo` tool. 
    * To fetch username and country when required use `retrieve_userinfo` tool.
    """,
    tools=[save_userinfo, retrieve_userinfo],  # Provide the tools to the agent
)

# Set up session service and runner
session_service = InMemorySessionService()
runner = Runner(agent=root_agent, session_service=session_service, app_name="default")

print("✅ Agent with session state tools initialized!")

✅ Agent with session state tools initialized!


### 5.3 实测 Session State

下面测试 Agent 如何利用 session state 在多轮对话中记住信息：

In [19]:
# Test conversation demonstrating session state
await run_session(
    runner,
    [
        "Hi there, how are you doing today? What is my name?",  # Agent shouldn't know the name yet
        "My name is Sam. I'm from Poland.",  # Provide name - agent should save it
        "What is my name? Which country am I from?",  # Agent should recall from session state
    ],
    "state-demo-session",
)


 ### Session: state-demo-session

User > Hi there, how are you doing today? What is my name?
gemini-2.5-flash-lite >  Hello! I'm doing great. I can't tell you your name just yet, as I haven't been told what it is. If you'd like to tell me, I can remember it for you!


User > My name is Sam. I'm from Poland.


gemini-2.5-flash-lite >  It's nice to meet you, Sam! I've saved your information.

User > What is my name? Which country am I from?


gemini-2.5-flash-lite >  Your name is Sam and you are from Poland.


### 5.4 查看 Session State

我们直接检查 session state，看里面存了什么：

In [20]:
# Retrieve the session and inspect its state
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="state-demo-session"
)

print("Session State Contents:")
print(session.state)
print("\n🔍 Notice the 'user:name' and 'user:country' keys storing our data!")

Session State Contents:
{'user:name': 'Sam', 'user:country': 'Poland'}

🔍 Notice the 'user:name' and 'user:country' keys storing our data!


### 5.5 Session State 隔离性

如前所示，session state 的关键属性是“按 session 隔离”。下面通过新建一个 session 来演示：

In [21]:
# Start a completely new session - the agent won't know our name
await run_session(
    runner,
    ["Hi there, how are you doing today? What is my name?"],
    "new-isolated-session",
)

# Expected: The agent won't know the name because this is a different session


 ### Session: new-isolated-session

User > Hi there, how are you doing today? What is my name?
gemini-2.5-flash-lite >  Hello! I'm doing great, thank you for asking. I don't have access to your name yet, as I haven't been provided with it. What is your name? 



### 5.6 跨 Session 的 State 共享

尽管 session 默认隔离，你可能会发现一些有趣现象。下面检查新 session（`new-isolated-session`）的 state：

In [22]:
# Check the state of the new session
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="new-isolated-session"
)

print("New Session State:")
print(session.state)

# Note: Depending on implementation, you might see shared state here.
# This is where the distinction between session-specific and user-specific state becomes important.

New Session State:
{'user:name': 'Sam', 'user:country': 'Poland'}


---

## 🧹 清理

In [23]:
# Clean up any existing database to start fresh (if Notebook is restarted)
import os

if os.path.exists("my_agent_data.db"):
    os.remove("my_agent_data.db")
print("✅ Cleaned up old database files")

✅ Cleaned up old database files


---
## 📊 总结

🎉 恭喜！你已经掌握构建 stateful AI agents 的基础能力：

- ✅ **Context Engineering** - 理解如何用 Context Compaction 组装 LLM 上下文
- ✅ **Sessions & Events** - 能在多轮对话中保持历史
- ✅ **Persistent Storage** - 让会话跨重启保留
- ✅ **Session State** - 能在对话中跟踪结构化数据
- ✅ **Manual State Management** - 体验了手动管理的能力与局限
- ✅ **Production Considerations** - 已具备应对真实场景挑战的思路

---

## ✅ 恭喜，完成本节 🎉

**ℹ️ 说明：无需提交！**

这个 Notebook 仅用于动手实践与学习。完成课程**不需要**把它提交到任何地方。

### 📚 继续学习

可参考以下文档：

- [ADK Documentation](https://google.github.io/adk-docs/)
- [ADK Sessions](https://google.github.io/adk-docs/)
- [ADK Session-State](https://medium.com/google-cloud/2-minute-adk-manage-context-efficiently-with-artifacts-6fcc6683d274)
- [ADK Session Compaction](https://google.github.io/adk-docs/context/compaction/#define-compactor)

### 🎯 下一步 - 长期记忆系统（Part 2）

#### 为什么需要 memory？
在本 Notebook 中，我们手动识别了少量特征（用户名、国家）并通过工具管理。但真实对话会涉及数百个特征：
- 用户偏好与习惯
- 历史交互及其结果
- 领域知识与熟练度
- 沟通风格与表达模式
- 主题之间的上下文关系

**ADK 的 Memory 系统会自动化这一整套过程**，是构建真正 Context-Aware Agent 的关键能力。

在下一份 Notebook（Part 2: Memory Management）中，你将学习：
- 如何自动从对话中抽取记忆
- 如何构建会随时间学习与适应的 Agent
- 如何规模化构建个性化体验
- 如何跨 session 管理长期知识

准备把手动 state 管理升级为智能自动化 Memory 系统了吗？继续 Part 2！